# Michigan Ave segment view

Per-run rows at corridor stops for **kept PIDs** (`michigan_ave_kept_pids.json`), with gap-filled `speed_mph` on the latitude-ordered northbound / southbound sequences.

Requires `data/mansueto/trips_{pid}_last30d.parquet` (see `scripts/cache_dlsd_parquets.py --source michigan --days 30`).

Output: `data/michigan_ave/segment_view.parquet` (gitignored).

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

_nb = Path.cwd() if (Path.cwd() / "michigan_ave_segment.py").is_file() else Path.cwd() / "notebooks"
if str(_nb) not in sys.path:
    sys.path.insert(0, str(_nb))

from michigan_ave_segment import (
    build_michigan_segment_view,
    direction_stop_sequences,
    format_segment_view_report,
    kept_pid_cache_status,
    kept_pids_path,
    load_kept_pids_meta,
    segment_view_report,
)

REPO_ROOT = _nb.parent if _nb.name == "notebooks" else _nb
OUT_DIR = REPO_ROOT / "data" / "michigan_ave"
OUT_PATH = OUT_DIR / "segment_view.parquet"
CACHE_VARIANT = load_kept_pids_meta()["cache_variant"]

In [ ]:
seqs = direction_stop_sequences()
print(f"Northbound corridor stops: {len(seqs['Northbound'])}")
print(f"Southbound corridor stops: {len(seqs['Southbound'])}")
meta = load_kept_pids_meta()
print(f"Kept PIDs: {meta['n_kept_pids']} ({kept_pids_path().name})")

In [ ]:
view = pd.DataFrame()
report = None
cache_status = kept_pid_cache_status(cache_variant=CACHE_VARIANT)
missing = cache_status.loc[~cache_status["cache_exists"], "pid"].tolist()
print(f"Cache files present: {cache_status['cache_exists'].sum()} / {len(cache_status)}")
if missing:
    print(f"Missing cache for PIDs: {', '.join(missing)}")

cache_ready = cache_status["cache_exists"].all()
if not cache_ready:
    print("\nStop: refresh Michigan Mansueto cache before building the view.")
    print("  python scripts/cache_dlsd_parquets.py --source michigan --days 30")
else:
    view = build_michigan_segment_view(cache_variant=CACHE_VARIANT)
    report = segment_view_report(view, cache_status=cache_status)
    print(format_segment_view_report(report))
    display(report["by_route"])
    display(report["by_pid"])
    if len(report["backwards_time_sample"]):
        display(report["backwards_time_sample"])
    display(view.head(10))

In [ ]:
if cache_ready and len(view):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    view.to_parquet(OUT_PATH, index=False)
    print(f"Wrote {OUT_PATH}")
else:
    print("Skipped writing segment_view.parquet (cache incomplete or empty view).")